In [1]:
# import
import pandas as pd
import os
import numpy as np

In [2]:
def safe_read_csv(filepath: str) -> pd.DataFrame:
    """
    여러 인코딩을 시도하여 CSV 파일을 안전하게 읽는 함수

    Parameters:
        filepath (str): CSV 파일 경로

    Returns:
        pd.DataFrame: 성공적으로 로드된 DataFrame
    """
    # 사용할 인코딩 후보 리스트
    encodings = ["cp949", "utf-8-sig", "utf-8", "euc-kr"]
    last_error = None  # 마지막으로 발생한 오류 저장

    # 후보 인코딩을 순서대로 시도
    for enc in encodings:
        try:
            # 주어진 인코딩으로 CSV 읽기
            df = pd.read_csv(filepath, encoding=enc)

            # 읽은 파일이 비어 있으면 오류 발생
            if df.empty:
                raise ValueError("CSV 파일이 비어 있습니다.")

            # 정상적으로 읽었으면 DataFrame 반환
            return df

        except Exception as e:
            # 실패하면 오류 기록 후 다음 인코딩 시도
            last_error = e
            continue

    # 모든 인코딩 시도 후에도 실패한 경우 예외 발생
    raise ValueError(f"CSV 로드 실패: {last_error}")


In [3]:
# 파일 내 첫 번째 컬럼명을 확인하고, 시군구 컬럼(법정동 정보)을 표준화, 정규화하는 함수
def integration_sgg_col(
    filepath: str
)-> pd.DataFrame:
    """
    주어진 CSV 파일의 첫 번째 컬럼명을 확인하여
    시군구 컬럼을 표준화·정규화한 뒤 result.csv로 저장

    Parameters:
        filepath (str): 입력 CSV 파일 경로

    Returns:
        pd.DataFrame: 시군구 컬럼 정제 및 표준화가 완료된 DataFrame
    """
    # 1) 인코딩 오류 방지하며 파일 불러오기
    df = safe_read_csv(filepath)
    first_col = df.columns[0] # 첫 번째 컬럼명
    print("완료")
    
    exit()
    # 2) 케이스 분기 
    if first_col == "시군구별(1)": # 법정동 컬럼 2개
        df = combine_region_columns(df) # 시군구 컬럼 2개 하나로 합치기(함수 제작 필요)
        df = replace_abbreviated_sido_names(df, column="시군구별", col_cnt=2) # 시도 치환
        df = merge_subdistricts_to_city(df, region_col="SGG_NAME") # 하위 행정구역 통합
    elif first_col =="시군구별": # 법정동 컬럼 1개
        df = replace_abbreviated_sido_names(df, column=first_col, col_cnt=1) # 시도 치환
        df = create_full_region_column(df, column="SGG_NAME") # 시군구_전체 컬럼 생성
    else: # 둘다 아닐때 데이터 초기 전처리 잘못되었으므로 오류 발생 내용 추가
        raise ValueError(
            f"지원하지 않는 첫 컬럼명: '{first_col}'. 허용: '시군구별(1)', '시군구별'"
        )
    
    df = merge_gunwigun(df, region_col="SGG_NAME") # 군위군 데이터 통합
    df = remove_target_regions(df, region_col="SGG_NAME") # 특례시 하위 행정구역 제거
    df = remove_only_sido_rows(df, region_col="SGG_NAME") # 시도만 있는 행 제거 
    df = remove_seoul_gyeonggi_incheon(df)
    df = check_missing_regions(df, region_col="SGG_NAME")
    validate_and_save_dataframe(df)
    
    # 이거 함수 구현
    df.to_csv("./result.csv", encoding="utf-8-sig", index=False)
    return df

In [4]:
def data_cleansing_folder(
    folder_path: str, 
    output_folder: str
)-> None:
    """
    모든 하위 폴더의 .csv 파일을 읽어 정제 후 output_folder에 저장

    Parameters:
        folder_path (str): 입력 CSV 파일들이 들어있는 최상위 폴더
        output_folder (str): 정제된 CSV를 저장할 최상위 폴더
    """
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".csv"):  # CSV 파일만 처리
                try:
                    # CSV 읽어서 시군구 컬럼 정제 (사용자 정의 함수)
                    df = integration_sgg_col(os.path.join(root, file))

                    # 원본 폴더 구조 보존: folder_path 기준 상대경로 계산
                    rel = os.path.relpath(root, folder_path)
                    save_dir = os.path.join(output_folder, rel)
                    os.makedirs(save_dir, exist_ok=True)  # 저장할 폴더 생성

                    # 저장 파일명: "_정리.csv" 붙이기
                    save_path = os.path.join(save_dir, file.replace(".csv", "_정리.csv"))

                    # 정제된 CSV 저장
                    df.to_csv(save_path, index=False, encoding="utf-8-sig")
                    print(f"✅ 저장 완료: {save_path}")

                except Exception as e:
                    # 오류 발생 시 로그 출력 (다른 파일 처리는 계속 진행)
                    print(f"❌ 오류 발생 {file}: {e}")

In [ ]:
# 수정 필요
input_path = "../../data/01-1_data-cleansing"
# 수정 필요
output_path = "../../data/01-2_data-cleansing"


# 나중에 하나
# 폴더 안 파일들 전체 데이터 정제
data_cleansing_folder(input_path, output_path)

완료
❌ 오류 발생 교원_1인당_학생수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 대학교 진학률_컬럼명변경.csv: name 'combine_region_columns' is not defined
완료
❌ 오류 발생 대학교_교원수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 대학교_수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 유치원_교원수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 유치원_수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 유치원_원아수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 인구_천명당_사설학원수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 초등학교_교원수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 초등학교_학생수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 학급당_학생수_컬럼명변경.csv: name 'replace_abbreviated_sido_names' is not defined
완료
❌ 오류 발생 남녀성비_시도_시_군_구__20250729160623.csv: name 'combine_region_columns' is not define

: 